# 01 — Data Preparation & Smoke Test

**But** : charger le dataset PatchCamelyon, vérifier les splits, et afficher quelques images pour valider que le pipeline de données fonctionne.

> Ce notebook est **indépendant** — il peut tourner après un redémarrage complet de runtime.

## 1. Setup (identique à chaque notebook)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/LAAFI_AI
except ImportError:
    pass

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

In [ ]:
import datasets
datasets.logging.set_verbosity_error()

from laafi_ai.config import ExperimentConfig
from laafi_ai.data import PCamDataModule
from laafi_ai.logging_utils import setup_logging
from laafi_ai.cli_train import set_seed

setup_logging()
config = ExperimentConfig.from_yaml('configs/default.yaml')
set_seed(config.seed)

## 2. Charger les DataLoaders

In [ ]:
data_module = PCamDataModule(config.data)
train_loader, val_loader, test_loader = data_module.dataloaders()

print(f'Train: {len(train_loader.dataset):,} images')
print(f'Val:   {len(val_loader.dataset):,} images')
print(f'Test:  {len(test_loader.dataset):,} images')

## 3. Smoke test — un batch

In [ ]:
images, labels = next(iter(train_loader))
print('Batch shape:', images.shape)
print('Label shape:', labels.shape)
print('Labels (10 premiers):', labels[:10])

## 4. Visualiser quelques images

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def denormalize_to_rgb(tensor):
    """Dé-normalise un tensor ImageNet pour affichage."""
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.permute(1, 2, 0).numpy()
    return np.clip(img * std + mean, 0, 1)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(denormalize_to_rgb(images[i]))
    label_text = 'Métastase' if labels[i] == 1 else 'Normal'
    ax.set_title(label_text, fontsize=9)
    ax.axis('off')
plt.suptitle('Échantillon du train set', fontsize=12)
plt.tight_layout()
plt.show()

---
**Prochain notebook** : `02_train.ipynb` pour lancer l'entraînement (ou reprendre depuis un checkpoint).